# Homework 26: Sparse Autoencoders for Transformer Features

**Audience.** Students who have completed the convolutional autoencoder assignment and inspected PicoGPT's residual stream.

**Prerequisites.** Autoencoders, MSE, ReLU, L1 regularization, transformer residual activations, and PyTorch training loops.

**Learning goals.** By the end, you will be able to:

- contrast an image autoencoder with an activation autoencoder;
- train an overcomplete sparse autoencoder (SAE);
- explain why the L1 penalty belongs on hidden feature activations;
- measure reconstruction error, sparsity, and dead features;
- test how SAE reconstruction affects the language model's loss.

The folder includes immutable fast reference artifacts so every calculation is reproducible. Before the course capstone, those should be supplemented with an instructor-trained TinyStories GPT and matching SAE for richer semantic features.


## From the previous autoencoder to an SAE

| Earlier denoising autoencoder | Sparse activation autoencoder |
|---|---|
| Reconstructs images | Reconstructs transformer activations |
| Compressed bottleneck | Overcomplete hidden representation |
| Dropout supplies corruption | L1 encourages few active features |
| Convolutional decoder | Learned feature directions |

Our SAE computes

\[
z=\operatorname{ReLU}(W_{enc}(h-b_{dec})+b_{enc}),\qquad
\hat h=zW_{dec}+b_{dec}.
\]

The loss is reconstruction MSE plus `lambda_l1 * mean(sum(abs(z), feature_axis))`. Summing within each example makes the sparsity cost comparable to the number of active features; averaging over all features would silently divide its strength by the SAE width.


In [1]:
# S1: Imports, course module, and reference checkpoint
from pathlib import Path
import sys

import torch
from torch import nn

if Path("pico_gpt.py").exists():
    COURSE_DIRECTORY = Path(".")
elif Path("Homework2026/pico_gpt.py").exists():
    COURSE_DIRECTORY = Path("Homework2026")
else:
    raise FileNotFoundError("Place pico_gpt.py beside this notebook.")
sys.path.insert(0, str(COURSE_DIRECTORY.resolve()))

from pico_gpt import (
    build_token_stream,
    get_batch,
    load_checkpoint,
    make_demo_stories,
    split_stories,
)

_ = torch.manual_seed(158)
tiny_reference_path = COURSE_DIRECTORY / "pico_gpt_tinystories_reference.pt"
fast_reference_path = COURSE_DIRECTORY / "pico_gpt_reference.pt"
checkpoint_path = (
    tiny_reference_path if tiny_reference_path.exists() else fast_reference_path
)
if not checkpoint_path.exists():
    raise FileNotFoundError(
        "The immutable course reference checkpoint is missing."
    )
model, tokenizer, metadata = load_checkpoint(checkpoint_path)

model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)

print("frozen GPT width:", model.config.d_model)
print("frozen GPT layers:", model.config.n_layers)
print("reference checkpoint:", checkpoint_path.name)


frozen GPT width: 64
frozen GPT layers: 2
reference checkpoint: pico_gpt_reference.pt


## 1. Collect residual-stream vectors

We train the SAE on an intermediate residual stream, with at least one transformer block remaining afterward. Each token position supplies one `d_model`-dimensional training example. PicoGPT remains frozen throughout.


In [2]:
# S2: Deterministically collect activation vectors and next-token labels
stories = make_demo_stories()
train_stories, validation_stories = split_stories(stories)
train_stream = build_token_stream(train_stories, tokenizer)
validation_stream = build_token_stream(validation_stories, tokenizer)
target_layer = max(0, model.config.n_layers - 2)
period_id = tokenizer.stoi["."]

@torch.no_grad()
def collect_activations(stream, number_of_batches, seed):
    generator = torch.Generator().manual_seed(seed)
    activation_batches = []
    period_label_batches = []

    for _ in range(number_of_batches):
        inputs, targets = get_batch(
            stream,
            batch_size=8,
            block_size=model.config.block_size,
            generator=generator,
        )
        _, _, cache = model(inputs, return_cache=True)
        residual = cache["blocks"][target_layer]["residual_post"]
        activation_batches.append(residual.reshape(-1, model.config.d_model))
        period_label_batches.append((targets == period_id).reshape(-1))

    return torch.cat(activation_batches), torch.cat(period_label_batches)

training_activations, training_period_labels = collect_activations(
    train_stream, number_of_batches=20, seed=260
)
validation_activations, validation_period_labels = collect_activations(
    validation_stream, number_of_batches=8, seed=261
)

print("training activations:", tuple(training_activations.shape))
print("validation activations:", tuple(validation_activations.shape))
print("fraction before a period:", round(float(validation_period_labels.float().mean()), 4))


training activations: (7680, 64)
validation activations: (3072, 64)
fraction before a period: 0.1406


## 2. The sparse autoencoder

The hidden width is four times the input width, so this is not a compressed bottleneck. Sparsity—not dimensionality reduction—forces the SAE to represent each activation using relatively few dictionary features.

We keep every decoder feature direction at unit norm. Without this constraint, the model could shrink the hidden activations to reduce the L1 penalty while enlarging the decoder weights to preserve the reconstruction.


In [3]:
# S3: A small overcomplete ReLU sparse autoencoder
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dimension, number_of_features):
        super().__init__()
        self.input_dimension = input_dimension
        self.number_of_features = number_of_features
        self.encoder = nn.Linear(input_dimension, number_of_features)
        self.decoder_directions = nn.Parameter(
            torch.empty(number_of_features, input_dimension)
        )
        self.decoder_bias = nn.Parameter(torch.zeros(input_dimension))

        nn.init.kaiming_uniform_(self.decoder_directions)
        self.normalize_decoder()
        with torch.no_grad():
            self.encoder.weight.copy_(self.decoder_directions)
            self.encoder.bias.zero_()

    def encode(self, activations):
        centered = activations - self.decoder_bias
        return torch.relu(self.encoder(centered))

    def forward(self, activations):
        features = self.encode(activations)
        reconstruction = features @ self.decoder_directions + self.decoder_bias
        return reconstruction, features

    @torch.no_grad()
    def normalize_decoder(self):
        norms = self.decoder_directions.norm(dim=1, keepdim=True).clamp_min(1e-8)
        self.decoder_directions.div_(norms)

sae = SparseAutoencoder(
    input_dimension=model.config.d_model,
    number_of_features=4 * model.config.d_model,
)

example_reconstruction, example_features = sae(training_activations[:10])
print("input shape:", tuple(training_activations[:10].shape))
print("feature shape:", tuple(example_features.shape))
print("reconstruction shape:", tuple(example_reconstruction.shape))
print("features are nonnegative:", bool(torch.all(example_features >= 0)))


input shape: (10, 64)
feature shape: (10, 256)
reconstruction shape: (10, 64)
features are nonnegative: True


## 3. Train the SAE

The GPT weights are frozen. Only the encoder, decoder directions, and decoder bias receive updates.


In [4]:
# S4: Reconstruction plus L1 sparsity
SAE_STEPS = 220
BATCH_SIZE = 256
LAMBDA_L1 = 0.01

optimizer = torch.optim.Adam(sae.parameters(), lr=2e-3)
generator = torch.Generator().manual_seed(262)
sae_history = []

for step in range(SAE_STEPS + 1):
    indices = torch.randint(
        len(training_activations),
        (BATCH_SIZE,),
        generator=generator,
    )
    batch = training_activations[indices]
    reconstruction, features = sae(batch)

    reconstruction_loss = ((reconstruction - batch) ** 2).mean()
    sparsity_loss = features.abs().sum(dim=-1).mean()
    total_loss = reconstruction_loss + LAMBDA_L1 * sparsity_loss

    if step % 55 == 0:
        sae_history.append((
            step,
            float(reconstruction_loss.detach()),
            float(sparsity_loss.detach()),
        ))
        print(
            f"step {step:3d} | reconstruction {reconstruction_loss:.5f} | "
            f"mean L1 per example {sparsity_loss:.5f}"
        )

    if step == SAE_STEPS:
        break
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    sae.normalize_decoder()


step   0 | reconstruction 0.38594 | mean L1 per example 35.73863
step  55 | reconstruction 0.04875 | mean L1 per example 5.67382
step 110 | reconstruction 0.03208 | mean L1 per example 4.92409
step 165 | reconstruction 0.02655 | mean L1 per example 4.27844
step 220 | reconstruction 0.02280 | mean L1 per example 4.19917


In [5]:
# S5: Evaluate reconstruction, sparsity, and dead features
sae.eval()
with torch.no_grad():
    validation_reconstruction, validation_features = sae(validation_activations)
    validation_mse = ((validation_reconstruction - validation_activations) ** 2).mean()
    activation_variance = ((validation_activations - validation_activations.mean(0)) ** 2).mean()
    normalized_mse = validation_mse / activation_variance
    mean_l0 = (validation_features > 1e-4).float().sum(dim=1).mean()
    dead_fraction = (validation_features.max(dim=0).values <= 1e-4).float().mean()
    decoder_norms = sae.decoder_directions.norm(dim=1)

print("validation MSE:", round(float(validation_mse), 6))
print("normalized MSE:", round(float(normalized_mse), 4))
print("mean active features (L0):", round(float(mean_l0), 2))
print("dead-feature fraction:", round(float(dead_fraction), 4))
print("decoder norm range:",
      round(float(decoder_norms.min()), 4),
      round(float(decoder_norms.max()), 4))


validation MSE: 0.022839
normalized MSE: 0.1596
mean active features (L0): 22.02
dead-feature fraction: 0.0
decoder norm range: 1.0 1.0


A useful SAE must balance reconstruction and sparsity. Zero features would be perfectly sparse but reconstruct badly. Activating every feature might reconstruct well but would not give a sparse description.


## 4. Does the reconstruction preserve language-model behavior?

MSE is not the final objective of PicoGPT. We therefore replace the selected residual stream with its SAE reconstruction and see how much next-token cross-entropy changes.


In [6]:
# S6: Substitute SAE reconstructions at the chosen transformer layer
evaluation_generator = torch.Generator().manual_seed(263)
evaluation_inputs, evaluation_targets = get_batch(
    validation_stream,
    batch_size=8,
    block_size=model.config.block_size,
    generator=evaluation_generator,
)

def reconstruct_residual(layer_index, residual):
    if layer_index != target_layer:
        return residual
    flat = residual.reshape(-1, residual.shape[-1])
    reconstructed, _ = sae(flat)
    return reconstructed.reshape_as(residual)

with torch.no_grad():
    _, baseline_lm_loss = model(evaluation_inputs, evaluation_targets)
    _, reconstructed_lm_loss = model(
        evaluation_inputs,
        evaluation_targets,
        intervention=reconstruct_residual,
    )

lm_loss_increase = reconstructed_lm_loss - baseline_lm_loss
print("baseline language-model loss:", round(float(baseline_lm_loss), 4))
print("with SAE reconstruction:", round(float(reconstructed_lm_loss), 4))
print("increase:", round(float(lm_loss_increase), 4))


baseline language-model loss: 0.6028
with SAE reconstruction: 0.7425
increase: 0.1397


## 5. Select a candidate feature on held-out activations

We do not assume that a fixed feature number means “period.” Separately trained SAEs can permute or split features. Instead, we select the feature with the largest held-out mean activation difference before periods versus elsewhere.


In [7]:
# S7: Select by held-out enrichment, then save the student's SAE
with torch.no_grad():
    period_feature_mean = validation_features[validation_period_labels].mean(dim=0)
    other_feature_mean = validation_features[~validation_period_labels].mean(dim=0)
    feature_scale = validation_features.std(dim=0).clamp_min(1e-6)
    feature_enrichment = (
        period_feature_mean - other_feature_mean
    ) / feature_scale
    selected_feature = int(feature_enrichment.argmax())

sae_checkpoint_path = COURSE_DIRECTORY / "pico_gpt_student_sae.pt"
torch.save(
    {
        "state_dict": sae.state_dict(),
        "input_dimension": sae.input_dimension,
        "number_of_features": sae.number_of_features,
        "target_layer": target_layer,
        "lambda_l1": LAMBDA_L1,
        "selected_feature": selected_feature,
    },
    sae_checkpoint_path,
)

print("selected feature:", selected_feature)
print("held-out enrichment:", round(float(feature_enrichment[selected_feature]), 5))
print("saved:", sae_checkpoint_path)


selected feature: 160
held-out enrichment: 2.53236
saved: pico_gpt_student_sae.pt


## Notebook checkpoints


In [8]:
# S8: Deterministic checkpoint record
checkpoint_26 = {
    "reference_checkpoint": checkpoint_path.name,
    "target_layer": target_layer,
    "training_activation_shape": tuple(training_activations.shape),
    "sae_feature_shape": tuple(example_features.shape),
    "sae_reconstruction_shape": tuple(example_reconstruction.shape),
    "features_nonnegative": bool(torch.all(example_features >= 0)),
    "decoder_norm_min": round(float(decoder_norms.min()), 4),
    "decoder_norm_max": round(float(decoder_norms.max()), 4),
    "normalized_mse": round(float(normalized_mse), 6),
    "mean_l0": round(float(mean_l0), 4),
    "dead_fraction": round(float(dead_fraction), 6),
    "lm_loss_increase": round(float(lm_loss_increase), 6),
    "selected_feature": selected_feature,
}
checkpoint_26


{'reference_checkpoint': 'pico_gpt_reference.pt',
 'target_layer': 0,
 'training_activation_shape': (7680, 64),
 'sae_feature_shape': (10, 256),
 'sae_reconstruction_shape': (10, 64),
 'features_nonnegative': True,
 'decoder_norm_min': 1.0,
 'decoder_norm_max': 1.0,
 'normalized_mse': 0.159642,
 'mean_l0': 22.0208,
 'dead_fraction': 0.0,
 'lm_loss_increase': 0.139693,
 'selected_feature': 160}

## Pitfall and extension

**Pitfall.** Saying “an L1 layer at the end” is misleading. L1 is a penalty on the SAE's hidden code `z`; the output remains a reconstruction of the original activation.

**Optional extension.** Train with three L1 strengths. Plot normalized reconstruction error against mean L0. This reconstruction–sparsity frontier is more informative than declaring one coefficient universally best.
